# Smart Training Center Management System

A console-based system for managing a training center's trainees, courses, attendance, scoring, and certificate eligibility — built with core Python and OOP.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import csv
import json
import tempfile

import numpy as np

print("Project environment is ready.")

## Constants


In [ ]:
DEFAULT_PASSING_SCORE = 60
DEFAULT_MIN_ATTENDANCE = 60
MIN_SCORE = 0
MAX_SCORE = 100

TRAINING_DAYS = (
    "Sunday", "Monday", "Tuesday", "Wednesday", "Thursday"
)

print("Training days:", TRAINING_DAYS)

## Custom Exceptions


In [ ]:
class DuplicateTraineeError(Exception):
    def __init__(self, trainee_id, message="Trainee with this ID is already registered"):
        self.trainee_id = trainee_id
        self.message = f"{message}: {trainee_id}"
        super().__init__(self.message)


class CourseFullError(Exception):
      def __init__(self, coures_id, message="Course with this ID is already fulled."):
        self.coures_id = coures_id
        self.message = f"{message}: {coures_id}"
        super().__init__(self.message)


class TraineeNotFoundError(Exception):
      def __init__(self, trainee_id, message="Trainee with this ID is not found."):
        self.trainee_id = trainee_id
        self.message = f"{message}: {trainee_id}"
        super().__init__(self.message)


class CourseNotFoundError(Exception):
      def __init__(self, trainee_id, message="Course with this ID is not found."):
        self.trainee_id = trainee_id
        self.message = f"{message}: {trainee_id}"
        super().__init__(self.message)


class InvalidScoreError(ValueError):
      def __init__(self, message="Invalid score."):
        self.message = f"{message}"
        super().__init__(self.message)

## Validation Functions


In [ ]:
def clean_name(name: str) -> str:
    no_spaces = name.strip()

    if no_spaces == "":
      raise ValueError("Name cannot be empty")

    no_double_spaces = " ".join(no_spaces.split())
    return no_double_spaces.title()


def validate_email(email: str) -> str:
    if email.count(" ") > 0:
        raise ValueError("Email must not contain spaces")
    if email.count("@") != 1:
        raise ValueError("Email must contain exactly one @")
    if email.find(".") < email.find("@"):
        raise ValueError("Email must contain a dot after @")
    return email

def validate_score(value: Any) -> float:
    converted_score = float(value)
    if converted_score < MIN_SCORE or converted_score > MAX_SCORE:
        raise InvalidScoreError()
    return converted_score


def validate_course_code(code: str) -> str:
    no_spaces = code.strip()
    upper_code = no_spaces.upper()
    if code == "":
        raise ValueError("Course code cannot be empty")
    return upper_code

## Validation Checks


In [ ]:
print(clean_name(" maha altalk "))
print(validate_email("maha@example.com"))
print(validate_score(88))
print(validate_course_code(" py101 "))

try:
    print(validate_email("maha@ example.com"))
    print(validate_score(120))
except (ValueError, InvalidScoreError):
    pass

## `Trainee` Class


In [ ]:
class Trainee:

    def __init__(
        self,
        trainee_id: int,
        name: str,
        email: str,
        enrolled_courses: set[str] | None = None,
        attendance: dict[str, set[int]] | None = None,
        scores: dict[str, list[float]] | None = None,
    ):
        self.trainee_id = int(trainee_id)
        self.name = clean_name(name)
        self.email = validate_email(email)

        self.enrolled_courses = set()
        if enrolled_courses:
            for code in enrolled_courses:
                validated_code = validate_course_code(code)
                self.enrolled_courses.add(validated_code)

        self.attendance = {}
        if attendance:
            for course, sessions in attendance.items():
                valid_course = validate_course_code(course)
                self.attendance[valid_course] = set(sessions)

        self.scores = {}
        if scores:
            for course, score_list in scores.items():
                valid_course = validate_course_code(course)
                self.scores[valid_course] = [validate_score(s) for s in score_list]

    def enroll(self, course_code: str) -> None:
        validated_code = validate_course_code(course_code)
        self.enrolled_courses.add(validated_code)

    def record_attendance(
        self,
        course_code: str,
        session_number: int,
    ) -> None:
        validated_code = validate_course_code(course_code)
        if validated_code in self.enrolled_courses:
            if validated_code not in self.attendance:
                self.attendance[validated_code] = set()
            self.attendance[validated_code].add(int(session_number))

    def add_score(
        self,
        course_code: str,
        score: float,
    ) -> None:
        validated_code = validate_course_code(course_code)
        validated_score = validate_score(score)
        if validated_code not in self.scores:
            self.scores[validated_code] = []
        self.scores[validated_code].append(validated_score)

    def average_score(self, course_code: str) -> float:
        validated_code = validate_course_code(course_code)
        if validated_code not in self.scores or not self.scores[validated_code]:
            return 0.0
        return sum(self.scores[validated_code]) / len(self.scores[validated_code])

    def to_dict(self) -> dict:
        enrolled_courses_list = []

        for course in self.enrolled_courses:
            enrolled_courses_list.append(course)

        attendance_dict = {}

        for course, sessions in self.attendance.items():
            attendance_dict[course] = []

            for session in sessions:
                attendance_dict[course].append(session)

        return {
            "trainee_id": self.trainee_id,
            "name": self.name,
            "email": self.email,
            "enrolled_courses": enrolled_courses_list,
            "attendance": attendance_dict,
            "scores": self.scores,
        }

    @classmethod
    def from_dict(cls, data: dict) -> "Trainee":
        return cls(
            trainee_id=data["trainee_id"],
            name=data["name"],
            email=data["email"],
            enrolled_courses=data.get("enrolled_courses", []),
            attendance=data.get("attendance", {}),
            scores=data.get("scores", {}),
        )

    def __str__(self) -> str:
        courses_str = ", ".join(self.enrolled_courses) if self.enrolled_courses else "None"
        return f"Trainee(ID: {self.trainee_id}, Name: {self.name}, Email: {self.email}, Courses: [{courses_str}])"

## `Course` Class


In [ ]:
class Course:
    def __init__(
        self,
        code: str,
        title: str,
        capacity: int,
        passing_score: float = DEFAULT_PASSING_SCORE,
        minimum_attendance: float = DEFAULT_MIN_ATTENDANCE,
        schedule: tuple[str, ...] = (),
    ):
        self.code = validate_course_code(code)
        self.title = title
        self.capacity = capacity
        self.passing_score = passing_score
        self.minimum_attendance = minimum_attendance
        self.schedule = schedule
        self.enrolled_trainees = set()

    def has_space(self) -> bool:
        if self.capacity > len(self.enrolled_trainees):
            return True
        return False

    def enroll_trainee(self, trainee_id: int) -> None:
        if self.has_space():
            self.enrolled_trainees.add(int(trainee_id))
        else:
            raise CourseFullError(self.code)

    def is_enrolled(self, trainee_id: int) -> bool:
        if int(trainee_id) in self.enrolled_trainees:
            return True
        return False

    def to_dict(self) -> dict:
        return {
            "code": self.code,
            "title": self.title,
            "capacity": self.capacity,
            "passing_score": self.passing_score,
            "minimum_attendance": self.minimum_attendance,
            "schedule": list(self.schedule),
            "enrolled_trainees": list(self.enrolled_trainees),
        }

    @classmethod
    def from_dict(cls, data: dict) -> "Course":
        course = cls(
            code=data["code"],
            title=data["title"],
            capacity=data["capacity"],
            passing_score=data.get("passing_score", DEFAULT_PASSING_SCORE),
            minimum_attendance=data.get("minimum_attendance", DEFAULT_MIN_ATTENDANCE),
            schedule=tuple(data.get("schedule", [])),
        )
        course.enrolled_trainees = set(data.get("enrolled_trainees", []))
        return course

    def __str__(self) -> str:
        return f"Course({self.code}: {self.title}, Capacity: {len(self.enrolled_trainees)}/{self.capacity})"

## `TrainingCenter` Class


In [ ]:
class TrainingCenter:
    def __init__(self, name: str):
        self.name = name
        self.trainees: dict[int, Trainee] = {}
        self.courses: dict[str, Course] = {}

    def add_course(self, course: Course) -> None:
        self.courses[course.code] = course

    def register_trainee(self, trainee: Trainee) -> None:
        if trainee.trainee_id in self.trainees:
            raise DuplicateTraineeError(trainee.trainee_id)
        self.trainees[trainee.trainee_id] = trainee

    def get_course(self, course_code: str) -> Course:
        if course_code not in self.courses:
            raise CourseNotFoundError(course_code)
        return self.courses[course_code]

    def get_trainee(self, trainee_id: int) -> Trainee:
        if trainee_id not in self.trainees:
            raise TraineeNotFoundError(trainee_id)
        return self.trainees[trainee_id]

    def enroll_trainee(
        self,
        trainee_id: int,
        course_code: str,
    ) -> None:
        trainee_id = int(trainee_id)
        course_code = validate_course_code(course_code)

        trainee = self.get_trainee(trainee_id)
        course = self.get_course(course_code)

        course.enroll_trainee(trainee_id)
        trainee.enroll(course_code)

    def record_attendance(
            self,
            trainee_id: int,
            course_code: str,
            session_number: int,
        ) -> None:
            trainee_id = int(trainee_id)
            session_number = int(session_number)

            trainee = self.get_trainee(trainee_id)
            course = self.get_course(course_code)

            trainee.record_attendance(course_code, session_number)


    def record_score(
          self,
          trainee_id: int,
          course_code: str,
          score: float,
      ) -> None:
          trainee_id = int(trainee_id)
          course_code = validate_course_code(course_code)
          score = float(score)
          self.get_trainee(trainee_id).add_score(course_code, score)

    def attendance_percentage(
          self,
          trainee_id: int,
          course_code: str
      ) -> float:
          trainee = self.get_trainee(trainee_id)
          course = self.get_course(course_code)
          attended = len(trainee.attendance.get(course_code, set()))
          total = len(course.schedule)
          return (attended / total * 100) if total > 0 else 0.0

    def course_status(
          self,
          trainee_id: int,
          course_code: str,
      ) -> str:
            trainee = self.get_trainee(trainee_id)
            if not trainee.scores.get(course_code):
                return "Incomplete"

            if trainee.average_score(course_code) < DEFAULT_PASSING_SCORE:
                return "Failed"
            return "Passed"

    def certificate_eligible(
        self,
        trainee_id: int,
        course_code: str,
    ) -> bool:

        trainee = self.get_trainee(trainee_id)
        course = self.get_course(course_code)

        has_scores = bool(trainee.scores.get(course_code))

        average_score = trainee.average_score(course_code)
        passed_score = average_score >= course.passing_score

        attendance_percentage = self.attendance_percentage(trainee_id, course_code)
        passed_attendance = attendance_percentage >= course.minimum_attendance

        if has_scores and passed_score and passed_attendance:
            return True
        else:
            return False

    def trainee_report(self, trainee_id: int) -> dict:
            trainee = self.get_trainee(trainee_id)
            courses_data = {}
            for c_code in trainee.enrolled_courses:
                courses_data[c_code] = {
                    "average": trainee.average_score(c_code),
                    "attendance": self.attendance_percentage(trainee_id, c_code),
                    "status": self.course_status(trainee_id, c_code),
                    "certificate_eligible": self.certificate_eligible(trainee_id, c_code)
                }
            return {
                "trainee_id": trainee.trainee_id,
                "name": trainee.name,
                "email": trainee.email,
                "courses": courses_data
            }
    def course_report(self, course_code: str) -> list[dict]:
          course = self.get_course(course_code)
          report = []
          for t_id in course.enrolled_trainees:
              if t_id in self.trainees:
                  trainee = self.trainees[t_id]
                  report.append({
                      "trainee_id": trainee.trainee_id,
                      "name": trainee.name,
                      "average_score": trainee.average_score(course_code),
                      "attendance_percentage": self.attendance_percentage(t_id, course_code),
                      "status": self.course_status(t_id, course_code)
                  })
          return report

    def overall_summary(self) -> dict:
        total_courses = len(self.courses)
        total_trainees = len(self.trainees)

        total_enrollments = 0
        passed_records = 0
        failed_records = 0
        incomplete_records = 0
        certificates_eligible = 0

        course_averages = {}

        for course_code, course in self.courses.items():
            total_enrollments += len(course.enrolled_trainees)

            averages = []

            for trainee_id in course.enrolled_trainees:
                status = self.course_status(trainee_id, course_code)

                if status == "Passed":
                    passed_records += 1
                elif status == "Failed":
                    failed_records += 1
                else:
                    incomplete_records += 1

                if self.certificate_eligible(trainee_id, course_code):
                    certificates_eligible += 1

                trainee = self.get_trainee(trainee_id)

                if trainee.scores.get(course_code):
                    averages.append(trainee.average_score(course_code))

            if averages:
                course_averages[course_code] = sum(averages) / len(averages)
            else:
                course_averages[course_code] = 0.0

        if course_averages:
            top_course_code = max(course_averages, key=course_averages.get)
            top_course = self.get_course(top_course_code).title
        else:
            top_course = "N/A"

        return {
            "center_name": self.name,
            "total_courses": total_courses,
            "total_trainees": total_trainees,
            "total_enrollments": total_enrollments,
            "passed_records": passed_records,
            "failed_records": failed_records,
            "incomplete_records": incomplete_records,
            "certificates_eligible": certificates_eligible,
            "top_course": top_course,
        }

    def save_json(self, path: Path) -> None:
          data = {
              "name": self.name,
              "trainees": [t.to_dict() for t in self.trainees.values()],
              "courses": [c.to_dict() for c in self.courses.values()],
          }
          path.write_text(json.dumps(data, indent=2), encoding="utf-8")

    @classmethod
    def load_json(cls, path: Path) -> "TrainingCenter":
          content = path.read_text(encoding="utf-8")
          data = json.loads(content)
          center = cls(data["name"])
          for t_data in data.get("trainees", []):
              center.register_trainee(Trainee.from_dict(t_data))
          for c_data in data.get("courses", []):
              center.add_course(Course.from_dict(c_data))
          return center

    def export_course_report_csv(
          self,
          course_code: str,
          path: Path,
      ) -> None:
          report = self.course_report(course_code)
          lines = ["trainee_id,name,average_score,attendance_percentage,status"]
          for row in report:
              lines.append(f"{row['trainee_id']},{row['name']},{row['average_score']},{row['attendance_percentage']},{row['status']}")
          path.write_text("\n".join(lines), encoding="utf-8")

## Course Catalog


In [ ]:
center = TrainingCenter("Future Skills Training Center")

course_1 = Course("PY101", "Python Fundamentals", 20, 70, 75, ("Mon", "Wed", "Fri"))
course_2 = Course("DA101", "Data Analysis with NumPy", 15, 80, 80, ("Tue", "Thu"))
course_3 = Course("OOP101", "Object-Oriented Programming", 10, 60, 70, ("Mon", "Wed"))
center.add_course(course_1)
center.add_course(course_2)
center.add_course(course_3)
print("Courses created:", len(center.courses))

## Register Trainees


In [ ]:
trainee_1 = Trainee(1111, "Sarah", "sara@example.com")
trainee_2 = Trainee(2222, "Omar", "omar@gmail.com")
trainee_3 = Trainee(3333, "Maha", "maha@gmail.com")
trainee_4 = Trainee(4444, "Haifa", "haifa@gmail.com")
trainee_5 = Trainee(5555, "Abdullah", "abdullah@gmail.com")

center.register_trainee(trainee_1)
center.register_trainee(trainee_2)
center.register_trainee(trainee_3)
center.register_trainee(trainee_4)
center.register_trainee(trainee_5)

print("Registered trainees:", len(center.trainees))
print("--------------------------")
print("Trainee 1:", trainee_1)
print("Trainee 2:", trainee_2)
print("Trainee 3:", trainee_3)
print("Trainee 4:", trainee_4)
print("Trainee 5:", trainee_5)

## Enrollment


In [ ]:
center.enroll_trainee(1111, "PY101")
center.enroll_trainee(2222, "OOP101")

center.enroll_trainee(2222, "PY101")
center.enroll_trainee(3333, "PY101")

center.enroll_trainee(2222, "DA101")
center.enroll_trainee(3333, "DA101")

center.enroll_trainee(4444, "PY101")

try:
    center.enroll_trainee(6666, "DA101")
except ValueError as e:
    print(e)
except TraineeNotFoundError as e:
    print(e)
except CourseNotFoundError as e:
    print(e)

small_course = Course("SQL99", "SQL", capacity=1, passing_score=60, minimum_attendance=60, schedule=("Sun",))
center.add_course(small_course)
center.enroll_trainee(1111, "SQL99")

try:
    center.enroll_trainee(2222, "SQL99")
except CourseFullError as e:
    print(e)

# Test missing course
try:
    center.enroll_trainee(1111, "AI999")
except CourseNotFoundError as e:
    print(e)

## Attendance Recording


In [ ]:
center.record_attendance(1111, "PY101", 1)
center.record_attendance(1111, "PY101", 2)
center.record_attendance(1111, "PY101", 3)

center.record_attendance(2222, "PY101", 1)
center.record_attendance(2222, "PY101", 1)  # duplicate session test
center.record_attendance(2222, "PY101", 2)

print("Attendance for Trainee 1111 in PY101 (%):", center.attendance_percentage(1111, "PY101"))
print("Attendance for Trainee 2222 in PY101 (%):", center.attendance_percentage(2222, "PY101"))
print("Attendance for Trainee 3333 in PY101 (%):", center.attendance_percentage(3333, "PY101"))

## Score Recording


In [ ]:
center.record_score(1111, "PY101", 85)
center.record_score(1111, "PY101", 90)

center.record_score(2222, "PY101", 75)
center.record_score(2222, "PY101", 80)

center.record_score(3333, "PY101", 40)
center.record_score(3333, "PY101", 50)

try:
    center.record_score(1111, "PY101", 120)
except InvalidScoreError as e:
    print(e)
print("Average score for Trainee 1111 in PY101:", center.get_trainee(1111).average_score("PY101"))
print("Average score for Trainee 2222 in PY101:", center.get_trainee(2222).average_score("PY101"))
print("Average score for Trainee 3333 in PY101:", center.get_trainee(3333).average_score("PY101"))

## NumPy Course Analysis


In [ ]:
selected_course_code = "PY101"

course_obj = center.get_course(selected_course_code)
averages = []

for t_id in course_obj.enrolled_trainees:
    trainee = center.get_trainee(t_id)

    if trainee.scores.get(selected_course_code):
        avg = trainee.average_score(selected_course_code)
        averages.append(avg)
scores_array = np.array(averages)

print(f"Statistics for course {selected_course_code}:")
print(f"Mean: {scores_array.mean():.2f}")
print(f"Minimum: {scores_array.min():.2f}")
print(f"Maximum: {scores_array.max():.2f}")
print(f"Standard Deviation: {scores_array.std():.2f}")

passing_averages = scores_array[scores_array >= course_obj.passing_score]
print(f"Passing averages (>= {course_obj.passing_score}):", passing_averages)

## Completion Status & Certificate Eligibility


In [ ]:
print("--- Course Status Tests ---")
print("Trainee 1111 Status:", center.course_status(1111, "PY101"))
print("Trainee 2222 Status:", center.course_status(2222, "PY101"))
print("Trainee 3333 Status:", center.course_status(3333, "PY101"))
print("Trainee 4444 Status:", center.course_status(4444, "PY101"))


print("\n--- Certificate Eligibility Results ---")
print("Trainee 1111 Eligible for Certificate:", center.certificate_eligible(1111, "PY101"))
print("Trainee 2222 Eligible for Certificate:", center.certificate_eligible(2222, "PY101"))
print("Trainee 3333 Eligible for Certificate:", center.certificate_eligible(3333, "PY101"))


## Trainee Report


In [ ]:
report = center.trainee_report(1111)
print(json.dumps(report, indent=2))

## Course Report


In [ ]:
course_code = "PY101"

report = center.course_report(course_code)

report = sorted(report, key=lambda trainee: trainee["average_score"], reverse=True)

print("----- Course Report -----")

for trainee in report:
    print("ID:", trainee["trainee_id"])
    print("Name:", trainee["name"])
    print("Average Score:", trainee["average_score"])
    print("Attendance:", trainee["attendance_percentage"])
    print("Status:", trainee["status"])
    print(
        "Certificate Eligible:",
        center.certificate_eligible(trainee["trainee_id"], course_code)
    )
    print("-------------------------")

## Overall Summary


In [ ]:
summary = center.overall_summary()
print(f"Training Center: {summary['center_name']}")
print(f"Courses: {summary['total_courses']}")
print(f"Registered trainees: {summary['total_trainees']}")
print(f"Total enrollments: {summary['total_enrollments']}")
print(f"Passed records: {summary['passed_records']}")
print(f"Failed records: {summary['failed_records']}")
print(f"Incomplete records: {summary['incomplete_records']}")
print(f"Certificates eligible: {summary['certificates_eligible']}")
print(f"Top course: {summary['top_course']}")

## JSON Save & Load


In [ ]:
with tempfile.TemporaryDirectory() as temp_dir:
    json_path = Path(temp_dir) / "training_center.json"

    center.save_json(json_path)
    print(f"Data successfully saved to {json_path}")

    loaded_center = TrainingCenter.load_json(json_path)
    print("Data successfully loaded into a new TrainingCenter object.")

    print("\n--- Original Center Summary ---")
    original_summary = center.overall_summary()
    for key, value in original_summary.items():
        print(f"{key}: {value}")

    print("\n--- Loaded Center Summary ---")
    loaded_summary = loaded_center.overall_summary()
    for key, value in loaded_summary.items():
        print(f"{key}: {value}")

    if original_summary == loaded_summary:
        print("\nVerification Passed: Original and loaded summaries match completely.")
    else:
        print("\nVerification Failed: Summaries do not match.")

## CSV Export


In [ ]:
with tempfile.TemporaryDirectory() as temp_dir:
    csv_path = Path(temp_dir) / "course_report.csv"

    course_code_to_export = "PY101"
    report = center.course_report(course_code_to_export)
    with open(csv_path, mode="w", newline="", encoding="utf-8") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow([
            "trainee_id",
            "trainee_name",
            "average_score",
            "attendance_percentage",
            "status",
            "certificate_eligible"
        ])

        for row in report:
            eligible = center.certificate_eligible(row["trainee_id"], course_code_to_export)
            writer.writerow([
                row["trainee_id"],
                row["name"],
                row["average_score"],
                row["attendance_percentage"],
                row["status"],
                eligible
            ])

    print(f"Course report successfully exported to {csv_path}\n")

    print("--- Reading Exported CSV File Contents ---")
    file_content = csv_path.read_text(encoding="utf-8")
    print(file_content)

## Menu-Driven Application


In [ ]:
def run_menu(center: TrainingCenter) -> None:
    while True:
        print("\n=== Smart Training Center Management System ===")
        print("1. List courses")
        print("2. List trainees")
        print("3. Register trainee")
        print("4. Enroll trainee")
        print("5. Record attendance")
        print("6. Record score")
        print("7. Show trainee report")
        print("8. Show course report")
        print("9. Show overall summary")
        print("10. Save data")
        print("0. Exit")

        choice_input = input("Enter your choice (0-10): ")

        if not choice_input.isdigit():
            print("Error: Please enter a valid numeric choice between 0 and 10.")
            continue

        choice = int(choice_input)

        try:
            if choice == 1:
                if not center.courses:
                    print("No courses available.")
                else:
                    for code, course in center.courses.items():
                        print(f"- [{code}] {course.title} (Capacity: {len(course.enrolled_trainees)}/{course.capacity})")

            elif choice == 2:
                if not center.trainees:
                    print("No registered trainees available.")
                else:
                    for tid, trainee in center.trainees.items():
                        print(f"- ID: {tid}, Name: {trainee.name}, Email: {trainee.email}")

            elif choice == 3:
                tid = int(input("Enter Trainee ID: ").strip())
                name = input("Enter Trainee Full Name: ").strip()
                email = input("Enter Trainee Email: ").strip()
                new_trainee = Trainee(tid, name, email)
                center.register_trainee(new_trainee)
                print(f"Success: Trainee '{name}' registered successfully.")

            elif choice == 4:
                tid = int(input("Enter Trainee ID: ").strip())
                ccode = input("Enter Course Code: ").strip()
                center.enroll_trainee(tid, ccode)
                print(f"Success: Trainee {tid} enrolled in course {ccode}.")

            elif choice == 5:
                tid = int(input("Enter Trainee ID: ").strip())
                ccode = input("Enter Course Code: ").strip()
                snum = int(input("Enter Session Number: ").strip())
                center.record_attendance(tid, ccode, snum)
                print(f"Success: Attendance recorded for session {snum}.")

            elif choice == 6:
                tid = int(input("Enter Trainee ID: ").strip())
                ccode = input("Enter Course Code: ").strip()
                score_val = float(input("Enter Score (0-100): ").strip())
                center.record_score(tid, ccode, score_val)
                print(f"Success: Score {score_val} recorded.")

            elif choice == 7:
                tid = int(input("Enter Trainee ID: ").strip())
                report = center.trainee_report(tid)
                print(json.dumps(report, indent=2))

            elif choice == 8:
                ccode = input("Enter Course Code: ").strip()
                report = center.course_report(ccode)
                print(f"\n--- Course Report: {ccode} ---")
                for row in report:
                    print(row)

            elif choice == 9:
                summary = center.overall_summary()
                print("\n--- Overall Management Summary ---")
                for k, v in summary.items():
                    print(f"{k}: {v}")

            elif choice == 10:
                save_path = Path("training_center_data.json")
                center.save_json(save_path)
                print(f"Success: Center data saved to {save_path.absolute()}")

            elif choice == 0:
                print("Exiting application. Goodbye!")
                break
            else:
                print("Invalid option. Please choose a number from the menu.")

        except (ValueError, TypeError) as err:
            print(f"Input Error: Please provide valid numeric values where required. Details: {err}")
        except (DuplicateTraineeError, CourseFullError, TraineeNotFoundError, CourseNotFoundError, InvalidScoreError) as err:
            print(f"Operation Error: {err}")
        except Exception as err:
            print(f"An unexpected error occurred: {err}")


## Test Suite


In [ ]:
def run_all_project_tests():
    print("=== Starting Project Test Suite ===")
    test_center = TrainingCenter("Test Management Center")

    # 1. Valid trainee registration
    try:
        t1 = Trainee(101, "Ali Ahmad", "ali@example.com")
        test_center.register_trainee(t1)
        assert 101 in test_center.trainees
        print("Test 1: Valid Trainee Registration -> PASS")
    except Exception as e:
        print(f"Test 1: Valid Trainee Registration -> FAIL ({e})")

    # 2. Invalid email
    try:
        validate_email("ali.example.com")
        print("Test 2: Invalid Email -> FAIL (Expected ValueError)")
    except ValueError:
        print("Test 2: Invalid Email -> PASS")

    # 3. Duplicate trainee
    try:
        t_dup = Trainee(101, "Duplicate Ali", "dup@example.com")
        test_center.register_trainee(t_dup)
        print("Test 3: Duplicate Trainee -> FAIL (Expected DuplicateTraineeError)")
    except DuplicateTraineeError:
        print("Test 3: Duplicate Trainee -> PASS")

    # Setup course for enrollment tests
    test_course = Course("TEST101", "Testing Fundamentals", capacity=1, schedule=("Sun", "Mon"))
    test_center.add_course(test_course)

    # 4. Valid enrollment
    try:
        test_center.enroll_trainee(101, "TEST101")
        assert test_course.is_enrolled(101)
        print("Test 4: Valid Enrollment -> PASS")
    except Exception as e:
        print(f"Test 4: Valid Enrollment -> FAIL ({e})")

    # 5. Missing course
    try:
        test_center.enroll_trainee(101, "FAKE999")
        print("Test 5: Missing Course -> FAIL (Expected CourseNotFoundError)")
    except CourseNotFoundError:
        print("Test 5: Missing Course -> PASS")

    # 6. Full course
    try:
        t2 = Trainee(102, "Sara Test", "sara@example.com")
        test_center.register_trainee(t2)
        test_center.enroll_trainee(102, "TEST101")
        print("Test 6: Full Course -> FAIL (Expected CourseFullError)")
    except CourseFullError:
        print("Test 6: Full Course -> PASS")

    # 7. Valid score
    try:
        score_val = validate_score(85.5)
        assert score_val == 85.5
        print("Test 7: Valid Score -> PASS")
    except Exception as e:
        print(f"Test 7: Valid Score -> FAIL ({e})")

    # 8. Invalid score below 0
    try:
        validate_score(-10)
        print("Test 8: Invalid Score Below 0 -> FAIL (Expected InvalidScoreError)")
    except InvalidScoreError:
        print("Test 8: Invalid Score Below 0 -> PASS")

    # 9. Invalid score above 100
    try:
        validate_score(110)
        print("Test 9: Invalid Score Above 100 -> FAIL (Expected InvalidScoreError)")
    except InvalidScoreError:
        print("Test 9: Invalid Score Above 100 -> PASS")

    # 10. Duplicate attendance record
    try:
        t1.record_attendance("TEST101", 1)
        t1.record_attendance("TEST101", 1)
        assert len(t1.attendance["TEST101"]) == 1
        print("Test 10: Duplicate Attendance Record -> PASS")
    except Exception as e:
        print(f"Test 10: Duplicate Attendance Record -> FAIL ({e})")

    # 11. Passed trainee & 12. Failed trainee & 13. Incomplete trainee
    try:
        c_status = Course("STAT101", "Statistics", capacity=5, passing_score=60, minimum_attendance=50, schedule=("S1", "S2"))
        test_center.add_course(c_status)

        tr_inc = Trainee(201, "Incomplete Guy", "inc@example.com")
        test_center.register_trainee(tr_inc)
        test_center.enroll_trainee(201, "STAT101")
        assert test_center.course_status(201, "STAT101") == "Incomplete"
        print("Test 13: Incomplete Trainee -> PASS")

        tr_pass = Trainee(202, "Passed Girl", "pass@example.com")
        test_center.register_trainee(tr_pass)
        test_center.enroll_trainee(202, "STAT101")
        test_center.record_score(202, "STAT101", 80.0)
        test_center.record_attendance(202, "STAT101", 1)
        assert test_center.course_status(202, "STAT101") == "Passed"
        print("Test 11: Passed Trainee -> PASS")

        tr_fail = Trainee(203, "Failed Guy", "fail@example.com")
        test_center.register_trainee(tr_fail)
        test_center.enroll_trainee(203, "STAT101")
        test_center.record_score(203, "STAT101", 40.0)
        assert test_center.course_status(203, "STAT101") == "Failed"
        print("Test 12: Failed Trainee -> PASS")
    except Exception as e:
        print(f"Test 11-13: Status Tests -> FAIL ({e})")

    # 14. JSON save and load
    try:
        with tempfile.TemporaryDirectory() as temp_dir:
            j_path = Path(temp_dir) / "test_center.json"
            test_center.save_json(j_path)
            restored_center = TrainingCenter.load_json(j_path)
            assert restored_center.name == test_center.name
            assert len(restored_center.trainees) == len(test_center.trainees)
            print("Test 14: JSON Save and Load -> PASS")
    except Exception as e:
        print(f"Test 14: JSON Save and Load -> FAIL ({e})")

    # 15. CSV report export
    try:
        with tempfile.TemporaryDirectory() as temp_dir:
            c_path = Path(temp_dir) / "test_report.csv"
            test_center.export_course_report_csv("TEST101", c_path)
            assert c_path.exists() and c_path.stat().st_size > 0
            print("Test 15: CSV Report Export -> PASS")
    except Exception as e:
        print(f"Test 15: CSV Report Export -> FAIL ({e})")

    print("=== All Tests Completed ===")

run_all_project_tests()